# آموزش واحد MoE: متن ChatGPT + logits معلم Qwen

این نوت‌بوک فقط یک مسیر آموزش دارد: **پاسخ‌های متنی ChatGPT + تقطیر sparse logits مدل معلم Qwen** در همان حلقهٔ آموزش مدل MoE. مسیرهای متن‌تنها و logits‌تنها اجرا نمی‌شوند.

آرشیو متنی از فایل نسخه‌بندی‌شدهٔ GitHub (`data/chatgpt_prompt_response_archive_part2.jsonl`) خوانده می‌شود. فقط خروجی logits و وضعیت/خروجی آموزش روی Google Drive ذخیره می‌شوند. ابتدا logits معلم برای همان توکن‌های پاسخ ذخیره می‌شود؛ سپس مدل پایه با ۲۴ checkpoint اولیهٔ MoE patch می‌شود و فقط FFNهای MoE/router آموزش می‌بینند. Backbone اصلی ثابت است.

**هدف محاسباتی واقعی:** با Qwen3.5-2B متراکم، فقط FFN فعالِ MoE حدود 56.6M پارامتر است؛ اما attention و projection خروجی واژگان همچنان متراکم‌اند و کل مسیر فعال حدود 1.09B می‌ماند. پس 86M را نمی‌توان ادعای کل پارامتر فعال دانست؛ این pipeline مقدار فعال FFN را جداگانه گزارش می‌کند. ابتدا گروه‌بندی کانال‌ها از وزن‌ها و فعال‌سازی‌های واقعی تحلیل می‌شود؛ سپس فقط یک آموزش واحدِ متن ChatGPT + sparse logits معلم اجرا می‌شود.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess, sys, json

REPO = Path('/content/Qwen-MoE-0.4B-')
WORK = Path('/content/drive/MyDrive/qwen-moe-chatgpt-logits')
INPUT_JSONL = REPO / 'data/chatgpt_prompt_response_archive_part2.jsonl'
LOGITS_OUT = WORK / 'qwen_logits_output'
OLD_CHECKPOINTS = Path('/content/drive/MyDrive/qwen-moe-full-build/checkpoints')
INITIAL_CHECKPOINTS = WORK / 'analysis_initialized_checkpoints'
ANALYSIS_DIR = WORK / 'architecture_analysis'
ANALYSIS_TEXT = WORK / 'analysis_corpus.txt'
EXPERT_PLAN = ANALYSIS_DIR / 'expert_channel_plan.json'
TRAIN_OUT = WORK / 'combined_moe_training'
MODEL_ID = 'Qwen/Qwen3.5-2B'
TOP_K = 64
TEMPERATURE = 1.0
MAX_SEQ_LENGTH = 256

WORK.mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/samanramezani1377-hub/Qwen-MoE-0.4B-.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers', 'accelerate', 'safetensors'], check=True)

required = {'ChatGPT text archive': INPUT_JSONL}
for label, path in required.items():
    if not path.exists():
        raise FileNotFoundError(f'{label} not found: {path}')

if not __import__('torch').cuda.is_available():
    raise RuntimeError('CUDA GPU is not available. Select a Colab GPU runtime before running training.')

print('Text archive:', INPUT_JSONL)
print('Teacher-logit output:', LOGITS_OUT)
print('Old checkpoints to clean up:', OLD_CHECKPOINTS)
print('New analysis-initialized checkpoints:', INITIAL_CHECKPOINTS)
print('Combined training output:', TRAIN_OUT)
print('Model:', MODEL_ID)
print('Pipeline: ChatGPT response CE + Qwen sparse-logit KD only')

In [ ]:
# پاک‌سازی محدود checkpointهای آزمایش قبلی؛ وزن‌های پایه، logits و سایر فایل‌ها دست‌نخورده می‌مانند.
import re
old_files = sorted(p for p in OLD_CHECKPOINTS.glob('layer_*_top2_32experts.pt')
                   if re.fullmatch(r'layer_\d{2}_top2_32experts\.pt', p.name))
old_layers = sorted(int(re.search(r'layer_(\d{2})', p.name).group(1)) for p in old_files)
if old_files:
    if old_layers != list(range(24)):
        raise RuntimeError(f'Expected exactly 24 old layer checkpoints (00..23); found {old_layers}. Nothing was deleted.')
    print('Deleting only these obsolete per-layer MoE checkpoints:')
    for p in old_files: print(' -', p)
    for p in old_files: p.unlink()
    print(f'Deleted {len(old_files)} obsolete checkpoints. Base Qwen weights and all other files were preserved.')
else:
    print('No old matching checkpoints found; nothing to delete.')

ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
rows = [json.loads(line) for line in INPUT_JSONL.read_text(encoding='utf-8').splitlines() if line.strip()]
ANALYSIS_TEXT.write_text('\n'.join(
    (str(row.get('prompt','')).strip() + ' ' + str(row.get('chatgpt_response','')).strip()).replace('\n',' ')
    for row in rows if row.get('prompt') and row.get('chatgpt_response')
) + '\n', encoding='utf-8')
print('Calibration documents:', len(rows), 'from', INPUT_JSONL)

# Step 1: analyze actual FFN weights and measured gate/up activations.
subprocess.run([
    sys.executable, str(REPO / 'scripts/analyze_expert_channels.py'),
    '--model', MODEL_ID, '--text-file', str(ANALYSIS_TEXT),
    '--output', str(EXPERT_PLAN), '--num-experts', '32',
    '--samples', str(min(256, len(rows))), '--max-length', str(MAX_SEQ_LENGTH), '--device', 'cuda'
], check=True)
plan = json.loads(EXPERT_PLAN.read_text(encoding='utf-8'))
TEACHER_REVISION = plan['teacher_revision']
print('Pinned Qwen revision:', TEACHER_REVISION)

# Step 2: initialize checkpoints from the analysis-derived groups; no separate training run.
subprocess.run([
    sys.executable, str(REPO / 'scripts/build_moe_initial_checkpoints.py'),
    '--model', MODEL_ID, '--plan', str(EXPERT_PLAN),
    '--output-dir', str(INITIAL_CHECKPOINTS), '--num-experts', '32', '--top-k', '2'
], check=True)
print('Analysis-derived initial checkpoints:', INITIAL_CHECKPOINTS)


## قالب دقیق آرشیو متن

فایل `data/chatgpt_prompt_response_archive_part2.jsonl` در همین ریپو قرار دارد و هر خط آن یک JSON object مستقل است و حداقل این فیلدها را داشته باشد:

- `sample_id`: شناسهٔ یکتا و پایدار
- `prompt`: پرسش/دستور ورودی
- `chatgpt_response`: پاسخ کامل، باکیفیت و آمادهٔ آموزش

نمونهٔ زیر فقط قالب را نشان می‌دهد و خودکار به فایل اضافه نمی‌شود. متن واقعی را از ایجنت تولیدکننده وارد کن.

In [ ]:
example = {
    "sample_id": "fa-reasoning-000001",
    "prompt": "چرا تقسیم بر صفر در حساب معمول تعریف نشده است؟",
    "chatgpt_response": "در تقسیم a بر b، عددی می‌خواهیم که ضرب آن در b برابر a شود. وقتی b صفر باشد، برای a غیرصفر چنین عددی وجود ندارد؛ و برای a برابر صفر نیز جواب یکتا نیست."
}
print(json.dumps(example, ensure_ascii=False, indent=2))
print('Example only; it is not appended to the training archive.')

## مرحلهٔ ۱ — استخراج logits معلم

اسکریپت توالی دقیق prompt+answer، توکن‌های هدف، logits برتر Qwen و log-normalizer کل واژگان را ذخیره می‌کند. log-normalizer برای حفظ جرم احتمالاتی خارج از top-k استفاده می‌شود. این KD یک تقریب با tail تجمیع‌شده است، نه KL دقیق روی تمام واژگان.

خروجی‌ها روی Drive ذخیره می‌شوند و اجرای دوباره فایل‌های tensor موجود را رد می‌کند. ورودی متن مستقیماً از فایل داخل ریپوی GitHub خوانده می‌شود و از آرشیو Google Drive استفاده نمی‌شود.

In [ ]:
subprocess.run([
    sys.executable, str(REPO / 'scripts/capture_qwen_logits_for_chatgpt_data.py'),
    '--input-jsonl', str(INPUT_JSONL),
    '--output-dir', str(LOGITS_OUT),
    '--model', MODEL_ID,
    '--revision', TEACHER_REVISION,
    '--top-k', str(TOP_K),
    '--temperature', str(TEMPERATURE),
    '--max-seq-length', str(MAX_SEQ_LENGTH),
    '--device', 'auto',
], check=True)

INDEX_JSONL = LOGITS_OUT / 'qwen_logits_index.jsonl'
MANIFEST_JSON = LOGITS_OUT / 'qwen_logits_manifest.json'
print('Index:', INDEX_JSONL)
print('Manifest:', MANIFEST_JSON)
print(json.loads(MANIFEST_JSON.read_text(encoding='utf-8'))) 

## مرحلهٔ ۲ — آموزش ترکیبی واحد

آموزش از ۲۴ checkpoint اولیهٔ MoE شروع می‌شود. مدل پایه با همان revision ثابت بارگذاری می‌شود، FFNهای dense با checkpointهای MoE جایگزین می‌شوند، وزن‌های backbone فریز می‌شوند و فقط expertها و routerها آموزش می‌بینند.

تابع هدف واحد است:

`total_loss = text_cross_entropy + 0.5 × qwen_sparse_KD`

Loss تقطیر از logits ذخیره‌شده استفاده می‌کند و جرم توکن‌های خارج از top-k را به‌صورت یک bucket تجمیع می‌کند. این نوت‌بوک هیچ اجرای text-only یا logits-only ندارد. تنظیمات پیش‌فرض یک epoch است؛ خروجی و وضعیت optimizer روی Drive ذخیره می‌شوند.

In [ ]:
TRAIN_OUT.mkdir(parents=True, exist_ok=True)
train_cmd = [
    sys.executable, str(REPO / 'scripts/train_moe_chatgpt_qwen_kd.py'),
    '--index-jsonl', str(INDEX_JSONL),
    '--checkpoint-dir', str(INITIAL_CHECKPOINTS),
    '--output-dir', str(TRAIN_OUT),
    '--model', MODEL_ID,
    '--revision', TEACHER_REVISION,
    '--num-experts', '32',
    '--top-k', '2',
    '--epochs', '1',
    '--learning-rate', '1e-5',
    '--kd-weight', '0.5',
    '--temperature', str(TEMPERATURE),
    '--save-every', '50',
    '--device', 'auto',
]
# If this run was interrupted, continue from the saved MoE checkpoint/optimizer state.
if (TRAIN_OUT / 'training_state.pt').is_file() and (TRAIN_OUT / 'checkpoints').is_dir():
    train_cmd.append('--resume')
print('Running the single combined training route.')
subprocess.run(train_cmd, check=True)

## مرحلهٔ ۳ — اعتبارسنجی ساختاری checkpoint نهایی

این مرحله فقط کامل‌بودن و سازگاری ساختاری ۲۴ checkpoint را بررسی می‌کند؛ ادعای بهبود کیفیت یا آمادگی استقرار نیست. برای ارزیابی کیفیت باید مجموعهٔ ارزیابی مستقل و از قبل کنارگذاشته‌شده استفاده شود؛ دادهٔ آموزش نباید به‌عنوان ارزیابی گزارش شود.

In [ ]:
VALIDATION_JSON = TRAIN_OUT / 'checkpoint_validation.json'
subprocess.run([
    sys.executable, str(REPO / 'scripts/validate_checkpoint_set.py'),
    '--checkpoint-dir', str(TRAIN_OUT / 'checkpoints'),
    '--layers', '24',
    '--num-experts', '32',
    '--top-k', '2',
    '--output', str(VALIDATION_JSON),
], check=True)
print(VALIDATION_JSON.read_text(encoding='utf-8'))
print('Training summary:')
print((TRAIN_OUT / 'training_summary.json').read_text(encoding='utf-8'))

## منبع داده و فایل‌های خروجی

- ورودی ثابت از GitHub: `data/chatgpt_prompt_response_archive_part2.jsonl` (۱٬۲۷۵ نمونه)\n- `qwen_logits_output/qwen_logits_index.jsonl` و پوشهٔ `qwen_topk_logits/`: سیگنال عددی معلم
- `combined_moe_training/checkpoints/`: checkpointهای نهایی آموزش‌شده
- `combined_moe_training/training_metrics.jsonl`: Loss و گرادیان هر مرحله
- `combined_moe_training/training_state.pt`: وضعیت optimizer برای ادامهٔ آموزش
- `combined_moe_training/training_summary.json`: خلاصهٔ اجرا

محدودیت‌ها: این نوت‌بوک و کد باید روی GPU واقعی اجرا شوند تا اجرای موفق تأیید شود. تقطیر top-k با tail تجمیع‌شده تقریب KL است. این روش به‌تنهایی تضمین کیفیت پاسخ یا اجرای سریع روی Android نیست.